# 05c — Noether Current: speak() (C)

**Requires the C kernel.** Install with `./install_c_kernel.sh`.  
Binary cells call `../../PtolC/ptolemy` via `system()`.

This notebook covers:
- `speak()`: computing the Noether current J^μ at each activated zero
- One-step A-propagation: J[j] += J[i] × min(A[i,j], 1/GAP) × β[j]
- The 1/GAP clamp and why it uses the same constant as learn()
- Recency weighting via λ decay
- Verbose speak output from the binary

Parallel Python notebook: `../05_noether_current_speaking.ipynb`

## 1. The Noether Current

```c
// Primary current at each activated zero
J[i] = β[i] × E[i]²

// One-step propagation through A
for each edge (i, j) with A[i,j] > 0:
    J[j] += J[i] × min(A[i,j], 1/GAP) × β[j]

// Response = top words by J, descending
```

The response is not generated by prediction. It is the Noether current — forced by conservation.

In [ ]:
#include <stdio.h>
#include <math.h>

#define GAP        0.000707
#define EMIT_THRESH 3.776

/* Simulate J^mu for a small field */
struct Zero {
    const char *word;
    int idx;
    double gamma;
    double E;
    double beta;
};

int main(void) {
    /* Approximate values from WordNet-trained checkpoint */
    struct Zero field[] = {
        {"water",       9361, 9331.32, 0.3663, 6.45},
        {"flow",        8820, 8812.10, 0.3600, 4.21},
        {"river",       7210, 7204.50, 0.3468, 3.89},
        {"stream",      6540, 6537.20, 0.3405, 3.55},
        {"downhill",    7890, 7901.50, 0.3520, 0.85},
        {"resistance",  4110, 4108.70, 0.3172, 2.10},
    };
    int n = 6;

    /* Primary Noether current */
    printf("Primary Noether current J[i] = beta[i] * E[i]^2:\n\n");
    printf("  %-12s  %8s  %8s  %10s  %s\n",
           "word", "beta", "E", "J[i]", "emit?");
    printf("  %-12s  %8s  %8s  %10s  %s\n",
           "------------", "--------", "--------", "----------", "-----");

    for (int i = 0; i < n; i++) {
        double J = field[i].beta * field[i].E * field[i].E;
        int emit = (field[i].beta >= EMIT_THRESH);
        printf("  %-12s  %8.4f  %8.4f  %10.6f  %s\n",
               field[i].word, field[i].beta, field[i].E, J,
               emit ? "YES" : "below thresh");
    }
    printf("\nEmission threshold = %.3f (|L_GROUND| x 2).\n", EMIT_THRESH);
    printf("Words below threshold are in the field but silent in speak().\n");
    return 0;
}

## 2. The 1/GAP Clamp

```c
J[j] += J[i] * min(A[i,j], 1.0/GAP) * beta[j]
```

`min(A, 1/GAP)` caps the propagation factor at `1/0.000707 ≈ 1414`.

The same GAP that prevents A-divergence in `learn()` prevents J^μ cascade in `speak()`.  
One problem. One constant. Two regulators.

In [ ]:
#include <stdio.h>
#include <math.h>

#define GAP  0.000707

int main(void) {
    double inv_GAP = 1.0 / GAP;

    printf("J^mu propagation clamp: min(A[i,j], 1/GAP)\n\n");
    printf("  1/GAP = %.2f   (maximum propagation factor)\n\n", inv_GAP);

    /* Show clamping at various A values */
    double A_vals[] = {0.001, 0.1, 10.0, 100.0, 500.0, 1000.0, 1414.0, 2000.0};
    printf("  %-10s  %12s  %s\n", "A[i,j]", "clamped to", "");
    printf("  %-10s  %12s\n", "----------", "------------");
    for (int i = 0; i < 8; i++) {
        double A    = A_vals[i];
        double Ac   = fmin(A, inv_GAP);
        int clamped = (A > inv_GAP);
        printf("  %-10.3f  %12.2f  %s\n", A, Ac,
               clamped ? "<-- CLAMPED" : "");
    }
    printf("\nWithout the clamp: large A[i,j] (near-zero pairs) would cascade.\n");
    printf("GAP in learn() prevents large A. GAP in speak() clamps any residual.\n");
    return 0;
}

## 3. Full Propagation Step

Simulate a complete J^μ propagation across a small A-graph:

In [ ]:
#include <stdio.h>
#include <math.h>
#include <string.h>

#define GAP          0.000707
#define EMIT_THRESH  3.776
#define MAX_N        8

struct Zero { const char *word; double E; double beta; };

/* A-edges: (from, to, weight) */
struct Edge { int i; int j; double w; };

int main(void) {
    struct Zero Z[MAX_N] = {
        {"water",    0.3663, 6.45},
        {"flow",     0.3600, 4.21},
        {"river",    0.3468, 3.89},
        {"stream",   0.3405, 3.55},
        {"path",     0.3210, 2.80},
        {"least",    0.3010, 1.95},
        {"downhill", 0.3520, 0.85},
        {"runs",     0.3290, 1.40},
    };
    int n = 8;

    /* Query: 'water flows' — activate water */
    /* A-edges built from WordNet co-occurrence */
    struct Edge edges[] = {
        {0, 1, 180.4},   /* water -> flow */
        {0, 2, 95.2},    /* water -> river */
        {0, 3, 72.1},    /* water -> stream */
        {1, 4, 44.8},    /* flow -> path */
        {2, 3, 210.5},   /* river -> stream */
        {3, 7, 33.2},    /* stream -> runs */
    };
    int ne = 6;

    /* Primary J */
    double J[MAX_N];
    for (int i = 0; i < n; i++)
        J[i] = Z[i].beta * Z[i].E * Z[i].E;

    printf("Primary J[i] = beta * E^2:\n");
    for (int i = 0; i < n; i++)
        printf("  %-10s  beta=%.2f  E=%.4f  J=%.4f\n",
               Z[i].word, Z[i].beta, Z[i].E, J[i]);

    /* One-step propagation */
    double J2[MAX_N];
    memcpy(J2, J, sizeof(J));
    for (int e = 0; e < ne; e++) {
        int i = edges[e].i, j = edges[e].j;
        double w = fmin(edges[e].w, 1.0 / GAP);
        J2[j] += J[i] * w * Z[j].beta;
    }

    printf("\nAfter one-step A-propagation:\n");
    printf("  %-10s  %10s  %10s  %s\n",
           "word", "J_primary", "J_after", "emit?");
    for (int i = 0; i < n; i++)
        printf("  %-10s  %10.4f  %10.4f  %s\n",
               Z[i].word, J[i], J2[i],
               Z[i].beta >= EMIT_THRESH ? "YES" : "-");

    printf("\nResponse = top words by J2, descending (above emit_thresh).\n");
    return 0;
}

## 4. Recency Weighting

β is monotone — it never forgets. Recency is handled in `speak()` via an age counter:

```c
w(age) = exp(-λ × age)   // λ = 0.05
```

A zero last activated 14 calls ago has weight ≈ 0.5. At 46 calls: ≈ 0.1. This biases J^μ toward recent context without modifying β.

In [ ]:
#include <stdio.h>
#include <math.h>

#define LAMBDA  0.05

int main(void) {
    printf("Recency weight w(age) = exp(-lambda * age)  [lambda=%.2f]:\n\n",
           LAMBDA);
    printf("  %-8s  %12s  %s\n", "age", "w(age)", "interpretation");
    printf("  %-8s  %12s  %s\n",
           "--------", "------------", "---------------");

    int ages[] = {0, 5, 10, 14, 20, 30, 46, 60, 100};
    for (int i = 0; i < 9; i++) {
        int age  = ages[i];
        double w = exp(-LAMBDA * age);
        const char *interp = "";
        if (age == 0)  interp = "just activated";
        if (age == 14) interp = "~half weight";
        if (age == 46) interp = "~tenth weight";
        printf("  %-8d  %12.6f  %s\n", age, w, interp);
    }
    printf("\nRecency applied in speak() only — beta is unchanged.\n");
    printf("The field remembers everything; recency weights what to say now.\n");
    return 0;
}

## 5. Live speak() Math Output

`-hv` shows the full J^μ pipeline in real time:

In [ ]:
#include <stdio.h>

#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    /* -hv: hear + speak with verbose math output */
    system(PTOLEMY " -hv 'water flows downhill' 2>/dev/null");
    return 0;
}

## Summary

- `speak()`: J[i] = β[i]×E[i]². Propagate via A: J[j] += J[i]×min(A,1/GAP)×β[j]. Sort descending.
- The 1/GAP clamp uses the same constant as the A-coupling denominator in `learn()` — one regulator, two functions.
- β is monotone. Recency lives in age counters, applied as exp(−λ×age) during J^μ computation.
- The response is not generated. It is the Noether current — conservation forced, not predicted.

**Next:** `06c_full_pipeline.ipynb` — complete pipeline using the ptolemy binary.